# RFMiD — construir DataFrames (train / val / test)

Objetivo: crear tres `pandas.DataFrame` listos para un `Dataset` de PyTorch, con:
- `image_path`: ruta a la foto de fondo de ojo (`.png`)
- `disease_risk`: etiqueta binaria (`Disease_Risk` del CSV)

Estructura del dataset:
- `Training_Set/Training_Set/RFMiD_Training_Labels.csv` + carpeta `Training/`
- `Evaluation_Set/Evaluation_Set/RFMiD_Validation_Labels.csv` + carpeta `Validation/`
- `Test_Set/Test_Set/RFMiD_Testing_Labels.csv` + carpeta `Test/`

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Localiza retina_clasificacion/ aunque el cwd del kernel sea el padre Retina/
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
if _cwd.is_dir():
    _candidates.extend(sorted(path for path in _cwd.iterdir() if path.is_dir()))
PROJECT_ROOT = next(
    directory
    for directory in _candidates
    if (directory / "src" / "config.py").is_file() and (directory / "requirements.txt").is_file()
)
sys.path = [entry for entry in sys.path if Path(entry or ".").resolve() != PROJECT_ROOT]
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_ROOT

# Datos RFMiD: pueden estar en PROJECT_ROOT o en el padre (Retina/)
SPLIT_CONFIG = {
    "train": {
        "labels_csv": DATA_ROOT / "Training_Set" / "Training_Set" / "RFMiD_Training_Labels.csv",
        "images_dir": DATA_ROOT / "Training_Set" / "Training_Set" / "Training",
    },
    "validation": {
        "labels_csv": DATA_ROOT / "Evaluation_Set" / "Evaluation_Set" / "RFMiD_Validation_Labels.csv",
        "images_dir": DATA_ROOT / "Evaluation_Set" / "Evaluation_Set" / "Validation",
    },
    "test": {
        "labels_csv": DATA_ROOT / "Test_Set" / "Test_Set" / "RFMiD_Testing_Labels.csv",
        "images_dir": DATA_ROOT / "Test_Set" / "Test_Set" / "Test",
    },
}

IMAGE_EXTENSION = ".png"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
for split_name, paths in SPLIT_CONFIG.items():
    print(
        f"{split_name}: csv={paths['labels_csv'].exists()}, "
        f"images_dir={paths['images_dir'].exists()}"
    )

## Función para construir cada DataFrame

Cada fila del CSV tiene un `ID` numérico; la imagen correspondiente es `{ID}.png` dentro de la carpeta del split.

In [ ]:
def build_split_dataframe(labels_csv: Path, images_dir: Path) -> pd.DataFrame:
    """Carga etiquetas RFMiD y resuelve la ruta de cada imagen de fondo de ojo."""
    labels = pd.read_csv(labels_csv)

    image_paths = [
        str(images_dir / f"{image_id}{IMAGE_EXTENSION}")
        for image_id in labels["ID"]
    ]

    split_dataframe = pd.DataFrame(
        {
            "image_path": image_paths,
            "disease_risk": labels["Disease_Risk"].astype(int),
        }
    )
    return split_dataframe


def count_missing_images(split_dataframe: pd.DataFrame) -> int:
    """Cuenta rutas que no apuntan a un fichero existente."""
    return sum(1 for path in split_dataframe["image_path"] if not Path(path).is_file())

## Construir los tres DataFrames

In [ ]:
train_df = build_split_dataframe(**SPLIT_CONFIG["train"])
validation_df = build_split_dataframe(**SPLIT_CONFIG["validation"])
test_df = build_split_dataframe(**SPLIT_CONFIG["test"])

datasets = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}

for split_name, split_dataframe in datasets.items():
    print(f"{split_name}: shape={split_dataframe.shape}")

## Vista previa y balance de clases

In [ ]:
for split_name, split_dataframe in datasets.items():
    print("=" * 60)
    print(f"Split: {split_name}")
    display(split_dataframe.head())

    class_counts = split_dataframe["disease_risk"].value_counts().sort_index()
    class_proportions = split_dataframe["disease_risk"].value_counts(normalize=True).sort_index()

    print("Conteo Disease_Risk (0=sin riesgo, 1=con riesgo):")
    print(class_counts.to_string())
    print("Proporciones:")
    print(class_proportions.round(3).to_string())

## Verificación: ¿existen las imágenes?

Comprobamos una muestra y el total de rutas faltantes por split.

In [ ]:
SAMPLE_SIZE = 10

for split_name, split_dataframe in datasets.items():
    sample_paths = split_dataframe["image_path"].head(SAMPLE_SIZE)
    sample_ok = all(Path(path).is_file() for path in sample_paths)
    missing_count = count_missing_images(split_dataframe)

    print(f"{split_name}:")
    print(f"  muestra de {SAMPLE_SIZE} IDs OK: {sample_ok}")
    print(f"  imágenes faltantes: {missing_count} / {len(split_dataframe)}")
    if missing_count > 0:
        missing_examples = [
            path for path in split_dataframe["image_path"]
            if not Path(path).is_file()
        ][:5]
        print("  ejemplos faltantes:", missing_examples)

print("\nListo: train_df, validation_df y test_df están disponibles en memoria.")